In [1]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [9]:
# TODO: Read pdf, feed into Claude

with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8") # images can be included as base64 encoding or a url to the image

messages = []
add_user_message(messages, [
    # Image Block
    {
        "type": "document", 
        "source": {
            "type": "base64",
            "media_type": "application/pdf", 
            "data": file_bytes
        },
        "title": "earth.pdf", # Add these 2 new fields to enable citation. The title field gives the document a readable name
        "citations": {"enabled": True} # Tells claude to track where it finds information
    },
    # Text Block
    {
        "type": "text",
        "text": 'How were Earths atmosphere and oceans formed?'
    }
]
)

In [10]:
chat(messages)

Message(id='msg_01W5oXjoHNKAHhmZ9N61bfdL', container=None, content=[TextBlock(citations=[CitationPageLocation(cited_text="[42]\r\nEarth's atmosphere and oceans were formed by volcanic activity and outgassing.\r\n", document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')], text="Earth's atmosphere and oceans were formed by volcanic activity and outgassing.", type='text'), TextBlock(citations=None, text=' ', type='text'), TextBlock(citations=[CitationPageLocation(cited_text='[43] Water vapor from\r\nthese sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets,\r\nand comets.\r\n', document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')], text='Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets.', type='text')], model='claude-sonnet-4-5-20250929', ro

In [4]:
article_text = """
After formation
Earth's atmosphere and oceans were formed by volcanic activity and outgassing.[43] Water vapor from
these  sources  condensed  into  the  oceans,  augmented  by  water  and  ice  from  asteroids,  protoplanets,
and comets.[44] Sufficient water to fill the oceans may have been on Earth since it formed.[45] In this
model, atmospheric greenhouse gases kept the oceans from freezing when the newly forming Sun had
only 70% of its current luminosity.[46] By 3.5 Ga, Earth's magnetic field was established, which helped
prevent the atmosphere from being stripped away by the solar wind.[47]

As the molten outer layer of Earth cooled it formed the first solid crust, which is thought to have been
mafic in composition. The first continental crust, which was more felsic in composition, formed by the
partial melting of this mafic crust.[49] The presence of grains of the mineral zircon of Hadean age in
Eoarchean sedimentary rocks suggests that at least some felsic crust existed as early as 4.4 Ga, only
140  Ma  after  Earth's  formation.[50]  There  are  two  main  models  of  how  this  initial  small  volume  of
continental  crust  evolved  to  reach  its  current  abundance:[51]  (1)  a  relatively  steady  growth  up  to  the
present day,[52] which is supported by the radiometric dating of continental crust globally and (2) an
initial  rapid  growth  in  the  volume  of  continental  crust  during  the  Archean,  forming  the  bulk  of  the
"""

In [ ]:
# Read plain text, feed into Claude
# Remove the pdf file open clause as we are reading straight from a plain text variable

messages = []
add_user_message(messages, [
    # Image Block
    {
        "type": "document", 
        "source": {
            "type": "text", # Change this from the pdf version
            "media_type": "text/plain",  # Change this from the pdf version
            "data": article_text # Source is now the variable
        },
        "title": "Earth plain text article", 
        "citations": {"enabled": True} 
    },
    # Text Block
    {
        "type": "text",
        "text": 'How were Earths atmosphere and oceans formed?'
    }
]
)

In [12]:
chat(messages)
# Note the citations is in CitationCharLocation

Message(id='msg_01QvibMkihZY278L3nocpNCV', container=None, content=[TextBlock(citations=[CitationCharLocation(cited_text="\nAfter formation\nEarth's atmosphere and oceans were formed by volcanic activity and outgassing.", document_index=0, document_title='Earth plain text artical', end_char_index=95, file_id=None, start_char_index=0, type='char_location')], text="Earth's atmosphere and oceans were formed by volcanic activity and outgassing.", type='text'), TextBlock(citations=None, text=' ', type='text'), TextBlock(citations=[CitationCharLocation(cited_text='[43] Water vapor from\nthese  sources  condensed  into  the  oceans,  augmented  by  water  and  ice  from  asteroids,  protoplanets,\nand comets.', document_index=0, document_title='Earth plain text artical', end_char_index=239, file_id=None, start_char_index=95, type='char_location')], text='Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets.', type='text'